# 1.0 - Exploración y Extracción de Features para P3 (Clustering de Estaciones)

**Autor:** AT (Ariadna Tapia)  
**Pregunta 3 (P3):** ¿Qué grupos de estaciones comparten patrones de uso y dónde se ubican en la ciudad?  
**Objetivo:** Explorar las distribuciones de viajes por estación, validar la lógica de agregaciones temporales/espaciales y consumir la función de ingeniería de características desde el módulo `mibici_mlops.featuresp3` respetando las buenas prácticas de Cookiecutter Data Science (la lógica vive en el módulo, el notebook es únicamente para exploración y validación visual).

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Añadir directorio raíz del proyecto si es necesario
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from mibici_mlops import config
from mibici_mlops.featuresp3 import generate_p3_station_features

sns.set_theme(style="whitegrid")
print("Configuración y módulos importados correctamente.")

## 1. Carga de Datos Procesados (Enero 2024 - Septiembre 2026)
Cargamos los datasets limpios y particionados en Apache Parquet desde `data/processed/`.

In [ ]:
trips_path = config.PROCESSED_DATA_DIR / "trips"
nomenclature_path = config.PROCESSED_DATA_DIR / "nomenclature"

print(f"Cargando viajes desde: {trips_path}")
trips_df = pd.read_parquet(trips_path)

print(f"Cargando nomenclatura desde: {nomenclature_path}")
stations_df = pd.read_parquet(nomenclature_path)

print(f"Total registros de viajes cargados: {len(trips_df):,}")
print(f"Total estaciones en catálogo: {len(stations_df):,}")

## 2. Inspección Exploratoria Inicial
Revisamos una muestra de viajes y del catálogo de nomenclatura para verificar esquemas y tipos de datos.

In [ ]:
display(trips_df.head())
display(stations_df.head())

## 3. Extracción de Features usando `featuresp3.py`
Invocamos directamente la función `generate_p3_station_features` del módulo para transformar el historial de viajes en un vector multidimensional por estación.

In [ ]:
station_features = generate_p3_station_features(trips_df, stations_df)
print(f"Matriz generada: {station_features.shape[0]} estaciones x {station_features.shape[1]} variables")
station_features.head(10)

## 4. Análisis Exploratorio de Variables (Distribución y Dispersión)
Analizamos los estadísticos descriptivos y las distribuciones de las características diseñadas para el clustering (ratios AM/PM, uso de fin de semana, rotación por anclaje).

In [ ]:
station_features.describe().T

In [ ]:
# Distribución de variables clave de movilidad
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.histplot(station_features["ratio_am"], kde=True, ax=axes[0, 0], color="royalblue")
axes[0, 0].set_title("Ratio Pico AM (Salidas / Entradas)")

sns.histplot(station_features["ratio_pm"], kde=True, ax=axes[0, 1], color="crimson")
axes[0, 1].set_title("Ratio Pico PM (Salidas / Entradas)")

sns.histplot(station_features["pct_fin_semana"], kde=True, ax=axes[0, 2], color="forestgreen")
axes[0, 2].set_title("Proporción de Uso en Fin de Semana")

sns.histplot(station_features["pct_domingo"], kde=True, ax=axes[1, 0], color="darkorange")
axes[1, 0].set_title("Proporción de Uso en Domingo (Vía RecreActiva)")

sns.histplot(station_features["rotacion_capacidad"], kde=True, ax=axes[1, 1], color="purple")
axes[1, 1].set_title("Rotación Relativa a Capacidad")

sns.scatterplot(
    data=station_features,
    x="longitude",
    y="latitude",
    hue="rotacion_capacidad",
    size="rotacion_capacidad",
    palette="viridis",
    ax=axes[1, 2]
)
axes[1, 2].set_title("Distribución Espacial de Estaciones")

plt.tight_layout()
plt.show()

## 5. Correlación entre Variables
Validamos que las variables extraídas aporten información complementaria y no redundante antes del modelado.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(station_features.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Matriz de Correlación de Features P3")
plt.show()

## 6. Validación de Consistencia para Modelado
Comprobamos que no existan valores nulos, infinitos ni estaciones desconectadas antes de que `trainp3.py` consuma estos datos para el pipeline de MLflow.

In [ ]:
null_counts = station_features.isnull().sum()
inf_counts = np.isinf(station_features).sum()

print("Valores Nulos:")
print(null_counts)
print("\nValores Infinitos:")
print(inf_counts)

assert null_counts.sum() == 0, "Existen valores nulos en la matriz de features."
assert inf_counts.sum() == 0, "Existen valores infinitos en la matriz de features."
print("\nMatriz de características validada con éxito. Lista para entrenamiento en modeling/trainp3.py.")